Часть 1. HDFS

1. Создать raw-слой и загрузить исходные данные

In [4]:
LOCAL_PRODUCT = "data/product.csv"
LOCAL_PC = "data/pc.csv"
HDFS_RAW_PRODUCT_DIR = "/warehouse/raw/product"
HDFS_RAW_PC_DIR = "/warehouse/raw/pc"

In [6]:
!hdfs dfs -mkdir -p {HDFS_RAW_PRODUCT_DIR}
!hdfs dfs -put -f {LOCAL_PRODUCT} {HDFS_RAW_PRODUCT_DIR}/

!hdfs dfs -mkdir -p {HDFS_RAW_PC_DIR}
!hdfs dfs -put -f {LOCAL_PC} {HDFS_RAW_PC_DIR}/

In [1]:
!hdfs dfs -ls /warehouse

Found 9 items
drwxrwxrwt   - hive  supergroup          0 2026-06-25 15:20 /warehouse/events_partitioned
drwxr-xr-x   - spark supergroup          0 2026-06-11 08:21 /warehouse/events_raw
drwxr-xr-x   - spark supergroup          0 2026-06-10 17:49 /warehouse/from_jupyter
drwxrwxrwt   - spark supergroup          0 2026-06-10 17:05 /warehouse/lab_check.db
drwxrwxrwt   - hive  supergroup          0 2026-06-25 15:20 /warehouse/lecture_03.db
drwxrwxrwt   - hive  supergroup          0 2026-06-25 15:40 /warehouse/ods
drwxrwxrwt   - hive  supergroup          0 2026-06-25 15:39 /warehouse/ods.db
drwxr-xr-x   - spark supergroup          0 2026-06-25 14:39 /warehouse/raw
drwxrwxrwt   - hive  supergroup          0 2026-06-25 15:11 /warehouse/raw.db


In [7]:
!hdfs dfs -ls {HDFS_RAW_PRODUCT_DIR}

Found 1 items
-rw-r--r--   1 spark supergroup        315 2026-06-25 14:39 /warehouse/raw/product/product.csv


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
spark = (
    SparkSession.builder
    .appName("homework-03")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/29 09:15:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Paths

In [3]:
BASE_PATH = "file:///materials/homework_03/data"

PRODUCT_PATH = f"{BASE_PATH}/product.csv"
PC_PATH = f"{BASE_PATH}/pc.csv"
LAPTOP_PATH = f"{BASE_PATH}/laptop.csv"
PRINTER_PATH = f"{BASE_PATH}/printer.csv"

## 5. Read CSV without type inference

Прочитайте CSV без автоматического определения типов и выведите схемы.

In [4]:
product = spark.read.option("header", True).option("inferSchema", False).csv(PRODUCT_PATH)
pc = spark.read.option("header", True).option("inferSchema", False).csv(PC_PATH)
laptop = spark.read.option("header", True).option("inferSchema", False).csv(LAPTOP_PATH)
printer = spark.read.option("header", True).option("inferSchema", False).csv(PRINTER_PATH)

In [ ]:
# числовые поля прочитались как `string`:

In [10]:
product.printSchema()

root
 |-- maker: string (nullable = true)
 |-- model: string (nullable = true)
 |-- type: string (nullable = true)



In [11]:
pc.printSchema()

root
 |-- model: string (nullable = true)
 |-- speed: string (nullable = true)
 |-- ram: string (nullable = true)
 |-- hd: string (nullable = true)
 |-- price: string (nullable = true)



In [12]:
laptop.printSchema()

root
 |-- model: string (nullable = true)
 |-- speed: string (nullable = true)
 |-- ram: string (nullable = true)
 |-- hd: string (nullable = true)
 |-- screen: string (nullable = true)
 |-- price: string (nullable = true)



In [13]:
printer.printSchema()

root
 |-- model: string (nullable = true)
 |-- color: string (nullable = true)
 |-- type: string (nullable = true)
 |-- price: string (nullable = true)



## 6. Cast types

Приведите типы и перезапишите те же переменные: `product`, `pc`, `laptop`, `printer`.

In [5]:
from pyspark.sql.functions import col

In [6]:
product = product.withColumn(
    "model", col("model").cast('int')
)
pc = (pc
    .withColumn(
    "speed", col("speed").cast('int'))
    .withColumn(
        'ram', col('ram').cast('int'))
    .withColumn(
        'hd', col('hd').cast('int'))
    .withColumn(
        'price', col('price').cast('int'))
)

In [7]:
laptop = (laptop
    .withColumn("model", col("model").cast("int"))
    .withColumn("speed", col("speed").cast("int"))
    .withColumn("ram", col("ram").cast("int"))
    .withColumn("hd", col("hd").cast("int"))
    .withColumn("screen", col("screen").cast("double"))
    .withColumn("price", col("price").cast("int"))
)

printer = (printer
    .withColumn("model", col("model").cast("int"))
    .withColumn("price", col("price").cast("int"))
)

In [8]:
product.printSchema()
pc.printSchema()
laptop.printSchema()
printer.printSchema()

root
 |-- maker: string (nullable = true)
 |-- model: integer (nullable = true)
 |-- type: string (nullable = true)

root
 |-- model: string (nullable = true)
 |-- speed: integer (nullable = true)
 |-- ram: integer (nullable = true)
 |-- hd: integer (nullable = true)
 |-- price: integer (nullable = true)

root
 |-- model: integer (nullable = true)
 |-- speed: integer (nullable = true)
 |-- ram: integer (nullable = true)
 |-- hd: integer (nullable = true)
 |-- screen: double (nullable = true)
 |-- price: integer (nullable = true)

root
 |-- model: integer (nullable = true)
 |-- color: string (nullable = true)
 |-- type: string (nullable = true)
 |-- price: integer (nullable = true)



## 7. Remove edge models

Выведите все строки `product`, кроме 3 моделей с наименьшими и 3 моделей с наибольшими номерами.

In [70]:
from pyspark.sql.functions import monotonically_increasing_id

product_sorted = product.orderBy('model')

product_with_n = product_sorted.withColumn('row_num', monotonically_increasing_id() + 1)
total_rows = product_sorted.count()


clean_product = product_with_n.filter(
    (col("row_num") > 3) & (col("row_num") <= total_rows - 3)
).drop("row_num")

clean_product.show()

+-----+-----+-------+
|maker|model|   type|
+-----+-----+-------+
|    A| 1004|Printer|
|    B| 1005|     PC|
|    B| 1006| Laptop|
|    B| 1007| Laptop|
|    C| 1008|     PC|
|    C| 1009|     PC|
|    C| 1010|     PC|
|    D| 1011|Printer|
|    D| 1012|Printer|
|    E| 1013|     PC|
|    E| 1014| Laptop|
|    F| 1015|     PC|
|    F| 1016| Laptop|
|    F| 1017|Printer|
|    G| 1018| Laptop|
|    H| 1019|     PC|
|    H| 1020|Printer|
|    I| 1021|     PC|
+-----+-----+-------+



## 8. Average PC price by speed

Для `speed > 600` посчитайте среднюю цену ПК по скорости.

In [12]:
from pyspark.sql.functions import (max, min, avg, sum, count, col)
filtered_speed = laptop.filter(col('speed')>600).groupBy('speed').agg(avg("price").alias('mean_price'))
filtered_speed.show()

[Stage 9:>                                                          (0 + 1) / 1]

+-----+----------+
|speed|mean_price|
+-----+----------+
| 1300|    1100.0|
| 1100|     950.0|
|  900|     700.0|
|  800|     680.0|
| 1000|     790.0|
| 1200|    1300.0|
+-----+----------+



## 9. Makers whose all PC models exist in PC

Найдите производителей ПК, все PC-модели которых из `product` есть в `pc`.

In [49]:
from pyspark.sql import functions as F
makers_with_models_not_in_pc = product.join(
    pc, 
    on = 'model',
    how = 'left_anti'
)

makers_with_models_in_pc = product.join(
    makers_with_models_not_in_pc, 
    on = 'maker',
    how = 'left_anti'
)
#makers_with_models_not_in_pc.show()
makers_with_models_in_pc.show()
makers_with_models_in_pc.select('maker').distinct().show()

+-----+-----+----+
|maker|model|type|
+-----+-----+----+
|    C| 1008|  PC|
|    C| 1009|  PC|
|    C| 1010|  PC|
+-----+-----+----+

+-----+
|maker|
+-----+
|    C|
+-----+



## 10. Laptop makers without printers

Найдите производителей, которые выпускают ноутбуки, но не выпускают принтеры.

In [22]:
from pyspark.sql import functions as F
product_lap_pr_count = product.groupby(['maker'])\
                                .agg(F.collect_set('type').alias('type_list'))\
                                .filter(
                                    (F.array_contains(F.col('type_list'), 'Laptop')) &
                                    ~(F.array_contains(F.col('type_list'), 'Printer'))
                                )
product_lap_pr_count.show()

+-----+------------+
|maker|   type_list|
+-----+------------+
|    E|[Laptop, PC]|
|    B|[Laptop, PC]|
|    J|    [Laptop]|
|    G|    [Laptop]|
+-----+------------+



## 11. All devices with price

Соберите DataFrame `model, device_type, price` из `pc`, `laptop`, `printer`. Найдите топ-5 самых дорогих устройств.

In [68]:
from pyspark.sql.functions import lit
full_df = pc.select('model', 'price').withColumn('device_type', lit('PC'))\
            .union(laptop.select('model', 'price').withColumn('device_type', lit('Laptop')))\
            .union(printer.select('model', 'price').withColumn('device_type', lit('Printer')))\
            
full_df.orderBy('price', ascending=False).show(5)

+-----+-----+-----------+
|model|price|device_type|
+-----+-----+-----------+
| 1016| 1300|     Laptop|
| 1010| 1250|         PC|
| 1007| 1100|     Laptop|
| 1006|  950|     Laptop|
| 1025|  900|         PC|
+-----+-----+-----------+
only showing top 5 rows



## 12. Price statistics by maker

Соедините `product` с объединённой таблицей цен и посчитайте статистику цен по производителю.

In [58]:
maker_price = product.join(
    full_df, on = 'model'
).groupBy('maker').agg(min('price').alias('min_price'),
                       max('price').alias('max_price'),
                       avg('price').alias('mean_price'))\
.orderBy('maker')
maker_price.show()

+-----+---------+---------+-----------------+
|maker|min_price|max_price|       mean_price|
+-----+---------+---------+-----------------+
|    A|      300|      780|            557.5|
|    B|      520|     1100|856.6666666666666|
|    C|      320|     1250|726.6666666666666|
|    D|      180|      220|            200.0|
|    E|      540|      680|            610.0|
|    F|      420|     1300|            860.0|
|    G|      430|      430|            430.0|
|    H|      250|      850|            550.0|
|    J|      790|      790|            790.0|
|    K|      190|      190|            190.0|
+-----+---------+---------+-----------------+



## 13. PC data quality check

Сравните PC-модели из `product` и таблицу `pc`. Найдите несовпадения.

In [67]:
pc_mismatch = pc.join(
    product, on = 'model', how = 'left_anti'
)
pr_mismatch = product.join(
    pc, on = 'model', how = 'left_anti'
)
pc_mismatch.show()
pr_mismatch.show()


+-----+-----+---+----+-----+
|model|speed|ram|  hd|price|
+-----+-----+---+----+-----+
| 1025| 1100| 16|1024|  900|
+-----+-----+---+----+-----+

+-----+-----+-------+
|model|maker|   type|
+-----+-----+-------+
| 1003|    A| Laptop|
| 1004|    A|Printer|
| 1006|    B| Laptop|
| 1007|    B| Laptop|
| 1011|    D|Printer|
| 1012|    D|Printer|
| 1014|    E| Laptop|
| 1015|    F|     PC|
| 1016|    F| Laptop|
| 1017|    F|Printer|
| 1018|    G| Laptop|
| 1020|    H|Printer|
| 1021|    I|     PC|
| 1022|    J| Laptop|
| 1023|    K|Printer|
| 1024|    L|     PC|
+-----+-----+-------+



## 14. Two cheapest models per type

Для каждого типа устройства найдите 2 модели с минимальной ценой.

In [74]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

windows = Window.partitionBy('device_type').orderBy('price')

full_df_with_n = full_df.withColumn(
    'row_num', F.row_number().over(windows)).filter(
    (col('row_num')<3))
full_df_with_n.show()

+-----+-----+-----------+-------+
|model|price|device_type|row_num|
+-----+-----+-----------+-------+
| 1018|  430|     Laptop|      1|
| 1014|  680|     Laptop|      2|
| 1008|  320|         PC|      1|
| 1001|  450|         PC|      2|
| 1011|  180|    Printer|      1|
| 1023|  190|    Printer|      2|
+-----+-----+-----------+-------+



## 15. Conditional aggregation by maker

Для каждого производителя посчитайте количество моделей `PC`, `Laptop`, `Printer`.

In [79]:
product.groupBy(['maker', 'type']).agg(
    count("*").alias('n')).orderBy(['maker', 'type']).show()

+-----+-------+---+
|maker|   type|  n|
+-----+-------+---+
|    A| Laptop|  1|
|    A|     PC|  2|
|    A|Printer|  1|
|    B| Laptop|  2|
|    B|     PC|  1|
|    C|     PC|  3|
|    D|Printer|  2|
|    E| Laptop|  1|
|    E|     PC|  1|
|    F| Laptop|  1|
|    F|     PC|  1|
|    F|Printer|  1|
|    G| Laptop|  1|
|    H|     PC|  1|
|    H|Printer|  1|
|    I|     PC|  1|
|    J| Laptop|  1|
|    K|Printer|  1|
|    L|     PC|  1|
+-----+-------+---+



## 16. Makers with all device types

Найдите производителей, у которых есть минимум один `PC`, один `Laptop` и один `Printer`.

In [86]:
product.groupby('maker').agg(F.collect_set('type').alias('types'))\
    .filter(F.size('types') == 3).orderBy('maker').show()

+-----+--------------------+
|maker|               types|
+-----+--------------------+
|    A|[Printer, Laptop,...|
|    F|[Printer, Laptop,...|
+-----+--------------------+



## 17. Bonus: write Parquet to HDFS

Запишите результат статистики цен по производителю в Parquet: `/warehouse/ods/maker_price_stats`.

In [91]:
maker_price.write.mode("overwrite").parquet("/warehouse/ods/maker_price_stats")

In [92]:
!hdfs dfs -ls /warehouse/ods

Found 2 items
drwxr-xr-x   - spark supergroup          0 2026-06-29 12:06 /warehouse/ods/maker_price_stats
drwxrwxrwt   - hive  supergroup          0 2026-06-25 15:44 /warehouse/ods/product
